# SLAM-ASR: Checkpoint Değerlendirme Aracı (Masaüstü & Colab Uyumlu)
### Eğitilmiş Tüm Modellerin Common Voice (CV-TR) Üzerinde Sıfır Maliyetli Hızlı Testi

Bu notebook, hem **kendi masaüstü bilgisayarınızda (GPU, Apple Silicon Mac veya CPU)** hem de **Google Colab** üzerinde çalışacak şekilde hibrit tasarlanmıştır.

* **Sıfır Maliyet / Sıfır Token:** Masaüstünüzde çalıştırırsanız tek kuruş ödemeden veya Colab tokenı harcamadan tüm modelleri değerlendirebilirsiniz.
* **Yeniden Eğitim:** Gerekmez. Önceden eğitilmiş ağırlıklar (`best_checkpoint`) doğrudan yüklenir.
* **Değerlendirme Süresi:** Model başına ortalama ~5-7 dakika.
* **Otomatik Kayıt:** Hesaplanan `eval_results_cv_generalization.jsonl` sonuçları ilgili model klasörüne kalıcı olarak kaydedilir.


In [ ]:
# 1. Donanım ve Çalışma Ortamı Kontrolü (Colab & Masaüstü Hibrit)
import os
import sys

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    print("[*] Google Colab ortamı tespit edildi.")
    !nvidia-smi
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
        print("[*] Google Drive başarıyla bağlandı.")
else:
    print("[*] Yerel Masaüstü Ortamı Tespit Edildi (Ücretsiz & Sıfır Token). 🚀")
    try:
        import torch
        if torch.cuda.is_available():
            print(f"[*] NVIDIA GPU Aktif: {torch.cuda.get_device_name(0)}")
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            print("[*] Apple Silicon GPU (MPS) Aktif.")
        else:
            print("[*] CPU Modunda Çalışılıyor.")
    except ImportError:
        print("[!] PyTorch henüz yüklü değil. Aşağıdaki hücre bağımlılıkları yükleyecektir.")


In [ ]:
# 2. Çalışma Dizinini Hazırlama ve Bağımlılıklar
import os
import sys

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    if not os.path.exists("slam-asr-reimplement"):
        !git clone https://github.com/gokhanersoy/slam-asr-reimplement.git
    %cd /content/slam-asr-reimplement/tr/tsc/t5-ng-ta-gormez-tail-aug-lora
    !pip install -q -r requirements.txt
    !pip install -q huggingface_hub soundfile torchaudio bitsandbytes accelerate datacollective mutagen peft
else:
    print(f"[*] Masaüstü Çalışma Dizini: {os.getcwd()}")
    # Gerekiyorsa bağımlılıkları yükleyebilirsiniz: pip install -r requirements.txt


In [ ]:
# 3. Mozilla Common Voice Test Verisini Hazırlama (Tek Seferlik)
import os
import json
import sys

cv_test_manifest = "data/cv/test.jsonl"
if not os.path.exists(cv_test_manifest):
    for cand in ["../cv/data/cv/test.jsonl", "../../cv/t0/data/cv/test.jsonl", "data/test.jsonl"]:
        if os.path.exists(cand):
            cv_test_manifest = cand
            break

def is_valid_manifest(path):
    if not path or not os.path.exists(path):
        return False
    try:
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                data = json.loads(line)
                audio_path = data.get("audio_filepath", "")
                return bool(audio_path and os.path.exists(audio_path))
    except Exception:
        return False
    return False

if not is_valid_manifest(cv_test_manifest):
    api_key = os.environ.get("MDC_API_KEY")
    if not api_key and "google.colab" in sys.modules:
        try:
            from google.colab import userdata
            api_key = userdata.get("MDC_API_KEY")
        except Exception:
            pass
    if not api_key:
        api_key = input("Lütfen Mozilla Data Collective API anahtarınızı (MDC_API_KEY) girin: ")
    
    print("[*] Common Voice test verisi indiriliyor ve hazırlanıyor...")
    !python scripts/prepare_common_voice.py \
        --download \
        --api_key "{api_key}" \
        --output_dir data/cv
    cv_test_manifest = "data/cv/test.jsonl"
else:
    print(f"[*] Common Voice test manifestosu ve ses dosyaları hazır: {cv_test_manifest}")


In [ ]:
# 4. Checkpoints Klasörünü Belirleme ve Modelleri Tarama
import glob
import os
import platform
import re
import json
import sys

IS_COLAB = "google.colab" in sys.modules

if IS_COLAB:
    checkpoints_root = "/content/drive/MyDrive/checkpoints"
else:
    # Google Drive yerel eşleme yollarını otomatik tara:
    # Windows (G:\Drive'ım\_Akademik_\Doktora (Gazi)\checkpoints), macOS veya yerel
    candidates = [
        # Windows: Doğrudan belirttiğiniz tam yol
        r"G:\Drive'ım\_Akademik_\Doktora (Gazi)\checkpoints",
        "G:/Drive'ım/_Akademik_/Doktora (Gazi)/checkpoints",
        "G:/Drive'im/_Akademik_/Doktora (Gazi)/checkpoints",
        r"G:\Drive'ım\checkpoints",
        "G:/Drive'ım/checkpoints",
        # macOS: CloudStorage (Google Drive for desktop - Drive'ım / _Akademik_)
        *sorted(glob.glob(os.path.expanduser("~/Library/CloudStorage/GoogleDrive*/Drive'ım/_Akademik_/Doktora (Gazi)/checkpoints"))),
        *sorted(glob.glob(os.path.expanduser("~/Library/CloudStorage/GoogleDrive*/Drive'im/_Akademik_/Doktora (Gazi)/checkpoints"))),
        *sorted(glob.glob(os.path.expanduser("~/Library/CloudStorage/GoogleDrive*/Drive'ım/checkpoints"))),
        *sorted(glob.glob(os.path.expanduser("~/Library/CloudStorage/GoogleDrive*/My Drive/checkpoints"))),
        os.path.expanduser("~/Google Drive/Drive'ım/_Akademik_/Doktora (Gazi)/checkpoints"),
        os.path.expanduser("~/Google Drive/Drive'ım/checkpoints"),
        "/Volumes/GoogleDrive/Drive'ım/_Akademik_/Doktora (Gazi)/checkpoints",
        "/Volumes/GoogleDrive/Drive'ım/checkpoints",
        # Yedek alternatifler
        "checkpoints",
        os.path.expanduser("~/Downloads/checkpoints"),
    ]
    checkpoints_root = None
    for cand in candidates:
        if os.path.exists(cand):
            checkpoints_root = cand
            break
    if not checkpoints_root:
        checkpoints_root = r"G:\Drive'ım\_Akademik_\Doktora (Gazi)\checkpoints" if platform.system() == "Windows" else os.path.expanduser("~/Google Drive/Drive'ım/_Akademik_/Doktora (Gazi)/checkpoints")

print(f"[*] Aranan Google Drive Checkpoints Dizini: {checkpoints_root}")

all_tsc_checkpoints = sorted(
    glob.glob(f"{checkpoints_root}/*slam_asr_tr_tsc*/best_checkpoint") +
    glob.glob(f"{checkpoints_root}/*/best_checkpoint"),
    key=lambda p: (re.search(r"(\d{8}_\d{6})", p).group(1) if re.search(r"(\d{8}_\d{6})", p) else "", p)
)

# Tekilleştirme
unique_ckpts = []
seen_folders = set()
for ckpt in all_tsc_checkpoints:
    parent = os.path.dirname(ckpt.rstrip("/"))
    if parent not in seen_folders:
        seen_folders.add(parent)
        unique_ckpts.append(ckpt)
all_tsc_checkpoints = unique_ckpts

print(f"[*] Toplam {len(all_tsc_checkpoints)} adet kontrol noktası tespit edildi:
")
print(f"{'No':<3} | {'Model Klasörü':<48} | {'CV Durumu':<14} | {'Mevcut WER / CER'}")
print("-" * 90)

checkpoint_status = []
for idx, ckpt in enumerate(all_tsc_checkpoints, 1):
    folder_dir = os.path.dirname(ckpt.rstrip("/"))
    folder_name = os.path.basename(folder_dir)
    eval_file = os.path.join(folder_dir, "eval_results_cv_generalization.jsonl")
    
    status = "EKSİK (Test Edilecek)"
    metrics = "-"
    if os.path.exists(eval_file):
        try:
            with open(eval_file, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        res = json.loads(line)
                        wer = res.get("metrics", {}).get("wer") or res.get("wer")
                        cer = res.get("metrics", {}).get("cer") or res.get("cer")
                        if wer is not None:
                            status = "TAMAMLANDI"
                            w_pct = wer * 100 if wer <= 1.0 else wer
                            c_pct = cer * 100 if cer <= 1.0 else cer
                            metrics = f"WER: {w_pct:.2f}%, CER: {c_pct:.2f}%"
        except Exception:
            status = "MEVCUT (Okunamadı)"
    
    checkpoint_status.append({
        "index": idx,
        "folder_name": folder_name,
        "folder_dir": folder_dir,
        "best_checkpoint": ckpt,
        "status": status,
        "metrics": metrics
    })
    print(f"{idx:<3} | {folder_name[:48]:<48} | {status:<14} | {metrics}")


In [ ]:
# 5. Eksik Olan Modelleri Sırayla Common Voice Üzerinde Değerlendirme
import subprocess
import shutil

# Tüm eksikleri otomatik test etmek için True bırakın.
# (İstediğiniz modelleri seçmek isterseniz: target_indices = [2, 4] gibi index numaraları yazabilirsiniz)
run_all_missing = True
target_indices = None

models_to_evaluate = []
for item in checkpoint_status:
    if target_indices is not None:
        if item["index"] in target_indices:
            models_to_evaluate.append(item)
    elif run_all_missing and "EKSİK" in item["status"]:
        models_to_evaluate.append(item)

print(f"[*] Toplam {len(models_to_evaluate)} model değerlendirilecek:\n")
for m in models_to_evaluate:
    print(f"  • [{m['index']}] {m['folder_name']}")

print("\n" + "=" * 75)

eval_script = "evaluate.py"
if not os.path.exists(eval_script):
    eval_script = "tr/tsc/t5-ng-ta-gormez-tail-aug-lora/evaluate.py"

for m in models_to_evaluate:
    folder_dir = m["folder_dir"]
    ckpt_path = m["best_checkpoint"]
    config_path = os.path.join(folder_dir, "hardware_adaptive_turkish.yaml")
    if not os.path.exists(config_path):
        config_path = "config/hardware_adaptive_turkish.yaml"
    
    output_file = os.path.join(folder_dir, "eval_results_cv_generalization.jsonl")
    
    print(f"\n[*] Değerlendirme Başlatılıyor: {m['folder_name']}")
    print(f"    Checkpoint:  {ckpt_path}")
    print(f"    Config:      {config_path}")
    print(f"    Hedef Çıktı: {output_file}")
    
    cmd = [
        sys.executable, eval_script,
        "--config", config_path,
        "--checkpoint", ckpt_path,
        "--manifest", cv_test_manifest,
        "--output_file", "temp_eval_cv.jsonl",
        "--language", "tr",
        "--beam_size", "4"
    ]
    
    ret = subprocess.run(cmd)
    if ret.returncode == 0 and os.path.exists("temp_eval_cv.jsonl"):
        shutil.copy("temp_eval_cv.jsonl", output_file)
        print(f"[✓] Başarılı! Sonuç kaydedildi: {output_file}")
    else:
        print(f"[!] {m['folder_name']} değerlendirmesi sırasında bir sorun oluştu.")

print("\n" + "=" * 75)
print("[*] Tüm seçilen modellerin değerlendirmesi tamamlandı!")


In [ ]:
# 6. Nihai Değerlendirme Tablosunu Yazdırma (Master Results Summary)
import json
import os

print("\n" + "=" * 75)
print("        TÜM MODELLERİN MOZILLA COMMON VOICE ÇAPRAZ TEST SONUÇLARI")
print("=" * 75)
print(f"{'Model Adı':<48} | {'WER (%)':<10} | {'CER (%)':<10}")
print("-" * 75)

for ckpt in all_tsc_checkpoints:
    folder_dir = os.path.dirname(ckpt.rstrip("/"))
    folder_name = os.path.basename(folder_dir)
    eval_file = os.path.join(folder_dir, "eval_results_cv_generalization.jsonl")
    
    wer_str = "-"
    cer_str = "-"
    if os.path.exists(eval_file):
        try:
            with open(eval_file, "r", encoding="utf-8") as f:
                for line in f:
                    if line.strip():
                        res = json.loads(line)
                        w = res.get("metrics", {}).get("wer") or res.get("wer")
                        c = res.get("metrics", {}).get("cer") or res.get("cer")
                        if w is not None:
                            w_pct = w * 100 if w <= 1.0 else w
                            wer_str = f"{w_pct:.2f}%"
                        if c is not None:
                            c_pct = c * 100 if c <= 1.0 else c
                            cer_str = f"{c_pct:.2f}%"
        except Exception:
            wer_str = "Hata"
    
    print(f"{folder_name[:48]:<48} | {wer_str:<10} | {cer_str:<10}")

print("=" * 75)
